# ClaimsSQL-Trust — generation on Kaggle

Runs **only the generate stage** of the benchmark for one model and saves the LLM
cache as `cache_<MODEL_ID>.zip`. Verification and scoring happen later on the laptop.

Before running: attach the dataset made from `claimssql_bundle.zip`, choose
**GPU T4 x2**, and turn **Internet on**. Then set the parameters below and use
**Save Version → Save & Run All**. Full steps are in `docs/KAGGLE.md`.

In [ ]:
# ---- Parameters -------------------------------------------------------------
MODEL_ID = "qwen"   # qwen | gemma | mid_oss   (keys in configs/models.yaml)
SPLIT = "all"       # all | dev | test
LIMIT = None        # e.g. 5 for a quick trial run; None = every question
RUN_ID = "full"     # only names the summary file

## 1. Unpack the bundle and install it

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from importlib.metadata import version
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
PROJECT = WORK / "claimssql"   # /kaggle/input is read-only, so work on a copy
CACHE = PROJECT / "results" / "cache"
OLLAMA_URL = "http://127.0.0.1:11434"


def find_bundle():
    """The bundle as uploaded (a zip) or as Kaggle unpacked it (a folder)."""
    zips = sorted(INPUT.rglob("claimssql_bundle.zip"))
    if zips:
        return zips[0]
    folders = [p.parent for p in sorted(INPUT.rglob("bundle_info.json"))
               if (p.parent / "pyproject.toml").exists()]
    if folders:
        return folders[0]
    raise FileNotFoundError(
        "Bundle not found under /kaggle/input. Attach the dataset made from "
        "claimssql_bundle.zip (Add Input)."
    )


bundle = find_bundle()
if PROJECT.exists():
    shutil.rmtree(PROJECT)
if bundle.is_file():
    with zipfile.ZipFile(bundle) as archive:
        archive.extractall(PROJECT)
else:
    shutil.copytree(bundle, PROJECT)

# Editable install: the code finds configs/ relative to its own folder.
# --upgrade: Kaggle's preinstalled packages can be older than the minimum
# versions in the bundle's pyproject.toml; this replaces them where needed.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "-e", str(PROJECT)], check=True
)
sys.path.insert(0, str(PROJECT / "src"))

for package in ("sqlglot", "duckdb", "pydantic", "httpx", "pyyaml"):
    print(f"{package:<9}{version(package)}")

bundle_info = json.loads((PROJECT / "bundle_info.json").read_text())
print("Bundle:", bundle)
print("Commit:", bundle_info["commit"])
print("Built :", bundle_info["created_utc"])
if bundle_info.get("uncommitted_changes"):
    print("WARNING: bundle was built with uncommitted changes:", bundle_info["uncommitted_changes"])

## 2. Resume from an earlier run

If the output of an earlier version of this notebook is attached as an input, its
cache is copied in first, so calls that were already made are not made again.

In [ ]:
CACHE.mkdir(parents=True, exist_ok=True)
copied = 0

# Cache zips as this notebook writes them.
for zip_path in sorted(INPUT.rglob("cache_*.zip")):
    with zipfile.ZipFile(zip_path) as archive:
        for name in archive.namelist():
            parts = name.split("/")
            if len(parts) == 3 and parts[0] == "cache" and name.endswith(".json"):
                target = CACHE / parts[1] / parts[2]
                if not target.exists():
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.write_bytes(archive.read(name))
                    copied += 1

# The same files if Kaggle unpacked the zip into folders.
for folder in sorted(INPUT.rglob("cache")):
    if folder.is_dir():
        for source in folder.glob("??/*.json"):
            target = CACHE / source.parent.name / source.name
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(source, target)
                copied += 1

print(f"Cached calls copied from earlier runs: {copied}")

## 3. Install Ollama and start the server

In [ ]:
# The installer unpacks a .tar.zst archive on recent versions; make sure zstd exists.
subprocess.run("apt-get install -y -qq zstd > /dev/null 2>&1", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

server_log = open(WORK / "ollama_serve.log", "w")
server = subprocess.Popen(
    ["ollama", "serve"],
    env=dict(os.environ, OLLAMA_HOST="127.0.0.1:11434"),
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

for _ in range(120):   # wait up to 2 minutes for the server to answer
    try:
        urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=2)
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError(
                "ollama serve exited; see /kaggle/working/ollama_serve.log"
            ) from None
        time.sleep(1)
else:
    raise RuntimeError("ollama serve did not respond within 2 minutes")
print("Ollama server is up")

## 4. Pull the model and check GPU memory

In [ ]:
from claimssql.config import load_models

model = load_models().get(MODEL_ID)   # the model tag comes from configs/models.yaml
assert model.provider == "ollama", f"{MODEL_ID} is not an Ollama model"
print(f"MODEL_ID {MODEL_ID} -> {model.model}, options {model.options}")

subprocess.run(["ollama", "pull", model.model], check=True)

# One tiny call with the benchmark's context size, so the memory shown below is
# what the real run will use.
warm_up = json.dumps({
    "model": model.model, "prompt": "hi", "stream": False,
    "options": {"num_predict": 1, "num_ctx": model.options.get("num_ctx", 8192)},
}).encode()
request = urllib.request.Request(f"{OLLAMA_URL}/api/generate", data=warm_up,
                                 headers={"Content-Type": "application/json"})
urllib.request.urlopen(request, timeout=900).read()

print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"],
    capture_output=True, text=True,
).stdout)

## 5. Check that cache keys will match the laptop

The bundle recorded the cache key of one probe call per model when it was built.
Recomputing it here must give the same value; otherwise the laptop would not find
the answers generated here, so the run stops before any model call.

In [ ]:
from claimssql.eval.fingerprint import differences, fingerprint

problems = differences(fingerprint(), bundle_info["fingerprint"])
if problems:
    raise RuntimeError("Cache keys differ from the bundle's: " + "; ".join(problems))
print("Cache-key fingerprint matches the bundle.")

## 6. Generate

In [ ]:
command = [sys.executable, "-u", "-m", "claimssql.eval.run_benchmark",
           "--stage", "generate", "--run-id", RUN_ID, "--models", MODEL_ID, "--split", SPLIT]
if LIMIT:
    command += ["--limit", str(LIMIT)]
print(" ".join(command))

started = time.time()
process = subprocess.Popen(
    command, cwd=PROJECT, env=dict(os.environ, OLLAMA_HOST=OLLAMA_URL),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in process.stdout:   # show progress lines as they arrive
    print(line, end="")
exit_code = process.wait()
elapsed_minutes = (time.time() - started) / 60
print(f"\nrun_benchmark exit code {exit_code} after {elapsed_minutes:.1f} minutes")

## 7. Save the cache

Always runs, also after a stopped or failed run, so finished calls are never lost.
The only files left in the output are `cache_<MODEL_ID>.zip`, the summary and the
Ollama log.

In [ ]:
summary_path = PROJECT / "results" / "runs" / f"{RUN_ID}.generate.json"
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}

cache_zip = WORK / f"cache_{MODEL_ID}.zip"
entries = sorted(CACHE.glob("??/*.json"))
with zipfile.ZipFile(cache_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in entries:
        archive.write(path, f"cache/{path.parent.name}/{path.name}")
    archive.write(PROJECT / "bundle_info.json", "bundle_info.json")
    archive.writestr("generate_summary.json", json.dumps(summary, indent=1))
(WORK / f"generate_summary_{MODEL_ID}.json").write_text(json.dumps(summary, indent=1))

server.terminate()
server_log.close()
shutil.rmtree(PROJECT)   # keep the notebook output small: the zip has everything

stats = summary.get(MODEL_ID, {})
print(f"Saved {cache_zip.name}: {len(entries)} cached calls, "
      f"{cache_zip.stat().st_size / 1e6:.1f} MB")
print(f"Model            : {stats.get('model')}")
print(f"Status           : {stats.get('status')}")
print(f"Questions done   : {stats.get('questions_done')} of {stats.get('questions')}")
print(f"Calls made here  : {stats.get('new_calls')}")
print(f"Cache hits       : {stats.get('cache_hits')}")
print(f"Seconds per call : {stats.get('seconds_per_new_call')}")
print(f"Commit           : {bundle_info['commit']}")

if exit_code != 0:
    raise RuntimeError(
        f"generate ended with exit code {exit_code} ({stats.get('status')}). The cache zip "
        "above holds every finished call: attach this version's output as an input and "
        "run again to continue."
    )